## Exercise 3: Install Libraries Notebook-Scoped

At HealthBridge Analytics, different teams sometimes need different versions of the same library. Rather than creating a new cluster for every variation, you can install libraries **notebook-scoped** — they apply only to the current notebook session and do not affect other users or notebooks on the same compute resource.

In this exercise, you install the `faker` library notebook-scoped and verify it works correctly.

### Task 3.1 — Install the `faker` library notebook-scoped

Your team uses `faker` to generate synthetic patient records for pipeline testing. Install it notebook-scoped so that it is available only in this notebook session.

> 🤖 **Databricks Assistant tip:** Use the prompt below to get started:
> *"How do I install a Python package notebook-scoped in Databricks using a magic command?"*

**Hint:** Use a `%pip` magic command to install the package. Pinning an exact version (e.g., `faker==37.1.0`) is a good practice for reproducibility.

In [ ]:
%pip install faker==37.1.0

### Task 3.2 — Verify the installation

Before using `faker` in a real pipeline, confirm it was installed correctly by:

1. Importing the `Faker` class from the `faker` library.
2. Creating a `Faker` instance.
3. Printing a randomly generated **full name** and **date of birth**.

> 🤖 **Databricks Assistant tip:**
> *"Show me a Python example of generating a random name and date of birth using the Faker library."*

In [ ]:
from faker import Faker

fake = Faker()

print("Full name   :", fake.name())
print("Date of birth:", fake.date_of_birth(minimum_age=18, maximum_age=85))

## Exercise 4: Generate and Analyze Synthetic Patient Data

Now that `faker` is installed, put it to work. Your team needs a synthetic dataset of patient admission records to test a new data pipeline. The data must look realistic enough to validate transformations and aggregations, but must contain **no real patient information** to comply with HIPAA policies.

In this exercise, you generate a synthetic dataset and run a basic analysis on it.

### Task 4.1 — Generate a synthetic patient admissions DataFrame

Use `faker` and the Spark session (`spark`) to create a **Spark DataFrame** with **100 synthetic patient admission records**. Each record must include:

| Column | Description |
|---|---|
| `patient_id` | Integer, 1 to 100 |
| `full_name` | Randomly generated full name |
| `date_of_birth` | Random date between `1940-01-01` and `2005-12-31`, as a string |
| `admission_date` | Random date between `2023-01-01` and `2025-12-31`, as a string |
| `diagnosis_code` | One of: `I21`, `J18`, `E11`, `K80`, `N39` (randomly chosen) |

Display the first 10 rows of the resulting DataFrame.

> 🤖 **Databricks Assistant tip:**
> *"How do I generate a list of Python dictionaries using the Faker library, then create a PySpark DataFrame from that list?"*
>
> **Hint:** Use `faker.date_of_birth()` with `minimum_age` and `maximum_age` parameters, or `faker.date_between()` with `start_date` and `end_date`. Use `random.choice()` for the diagnosis code.

In [ ]:
import random
from faker import Faker
from datetime import date

fake = Faker()

diagnosis_codes = ["I21", "J18", "E11", "K80", "N39"]

records = []
for i in range(1, 101):
    records.append({
        "patient_id": i,
        "full_name": fake.name(),
        "date_of_birth": str(fake.date_between(start_date=date(1940, 1, 1), end_date=date(2005, 12, 31))),
        "admission_date": str(fake.date_between(start_date=date(2023, 1, 1), end_date=date(2025, 12, 31))),
        "diagnosis_code": random.choice(diagnosis_codes)
    })

df = spark.createDataFrame(records)
display(df.limit(10))

### Task 4.2 — Analyze admissions by diagnosis code

The clinical informatics team wants to know which diagnosis codes appear most frequently in the admission data. Using the Spark DataFrame you created in Task 4.1:

1. Count the number of admissions per `diagnosis_code`.
2. Order the results from **most to least** admissions.
3. Display the result.

> 🤖 **Databricks Assistant tip:**
> *"How do I group by a column, count occurrences, and sort the result in descending order using PySpark DataFrame API?"*

In [ ]:
from pyspark.sql.functions import count, desc

admission_counts = (
    df
    .groupBy("diagnosis_code")
    .agg(count("*").alias("admission_count"))
    .orderBy(desc("admission_count"))
)

display(admission_counts)